# 05 · Validación y evidencias

## 1. Trazabilidad Bronze → Silver → Gold

In [1]:
"""
Aquí armamos la tabla de evidencias que une las tres capas: para cada archivo mostramos cuántas filas tiene, su sha256 y la fecha
de la fuente. En Bronze tomamos las cifras del manifiesto, que fue escrito en el momento de la descarga, y en Silver y Gold las
recalculamos leyendo los archivos. Lo hicimos así porque queríamos que la cadena se pudiera auditar de punta a punta sin confiar en lo
que dice un documento. La salida muestra que las 5.146 construcciones entran por Bronze y llegan a Gold sin pérdida. Eso prueba
que no perdimos filas por el camino, pero no prueba que cada transformación sea correcta, eso lo cubren las pruebas y el EDA.
"""
import sys, json, subprocess, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd
from src.common import ROOT, load_config, read_manifest, sha256_file

cfg = load_config()
pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70, 'display.max_rows', 80)
man = pd.DataFrame(read_manifest())
bronze = man.assign(capa='Bronze', archivo=man.path.str.split('/').str[-1], sha=man.sha256.str[:12] + '…', fecha_fuente=man.fetched_at_utc)[['capa', 'archivo', 'rows', 'sha', 'fecha_fuente']]
filas = []
for nombre in ('construcciones', 'estrato_barrio', 'integrated', 'orphans_left', 'orphans_right'):
    p = ROOT / cfg['paths_silver'][nombre]
    filas.append(('Silver', p.name, len(pd.read_csv(p)), sha256_file(p)[:12] + '…', 'derivado de Bronze'))
for p in ('gold_final.csv', 'kpis_gold.csv', 'kpis_sensibilidad.csv'):
    f = ROOT / 'data/gold' / p
    filas.append(('Gold', p, len(pd.read_csv(f)), sha256_file(f)[:12] + '…', 'fecha_corte ' + cfg['fechas']['fecha_edicion_capa']))
tabla = pd.concat([bronze.rename(columns={'rows': 'filas'}), pd.DataFrame(filas, columns=['capa', 'archivo', 'filas', 'sha', 'fecha_fuente'])], ignore_index=True)
print(tabla.to_string(index=False))
print('\nBronze intacto contra el manifiesto:', all(sha256_file(ROOT / m['path']) == m['sha256'] for m in read_manifest()))
print('páginas de la API suman el conteo oficial:', int(man[man.source_id == 'edificaciones'].rows.sum()), '=', int(man[man.source_id == 'edificaciones_count_antes'].rows.iloc[0]))


  capa                                                                   archivo  filas           sha           fecha_fuente
Bronze                                                          count_antes.json   5146 c20dec67463c…   2026-10-16T00:10:24Z
Bronze                                                            capa_meta.json     20 c5e4f4258775…   2026-10-16T00:10:25Z
Bronze                                                            page_0001.json   2000 92dfda868878…   2026-10-16T00:10:25Z
Bronze                                                            page_0002.json   2000 6b2a4849e95d…   2026-10-16T00:10:26Z
Bronze                                                            page_0003.json   1146 ac23590d1a67…   2026-10-16T00:10:27Z
Bronze                                                        count_despues.json   5146 c20dec67463c…   2026-10-16T00:10:27Z
Bronze estratificaci_n_socioecon_mica_seg_n_estrato__por_barrio_en_cali_2015.csv    369 c16b209e4f20…   2026-10-16T00:10:28Z


## 2. Cobertura y calidad del JOIN

In [2]:
"""
En esta celda mostramos las tres coberturas de K5 con su numerador y denominador, tomadas del reporte de calidad de Silver y de
kpis_gold.csv, y comprobamos que ambas fuentes digan lo mismo. Hicimos esto porque la cobertura de llave es el criterio que pide el
enunciado, con umbral de 80 %, mientras que la cobertura de estrato válido y el porcentaje de nombre validado son controles propios
que nos parecieron necesarios. La salida muestra que el umbral se cumple de sobra. No demuestra que el barrio asignado sea el
correcto en cada construcción: la validación por nombre sube la confianza de la llave, pero no la vuelve perfecta.
"""
rep = json.loads((ROOT / cfg['paths_silver']['quality_report']).read_text(encoding='utf-8'))
kp = pd.read_csv(ROOT / 'data/gold/kpis_gold.csv'); sens = pd.read_csv(ROOT / 'data/gold/kpis_sensibilidad.csv')
k5 = kp[kp.kpi_id == 'K5'].set_index('grupo')
print(k5[['numerador', 'denominador', 'valor']].round(6))
print('umbral:', rep['umbral_cobertura_join'], '->', rep['resultado_umbral'], '| llave_confianza:', rep['llave_confianza'])
print('K5 de Gold coincide con quality_report.json:', abs(k5.loc['cobertura_llave', 'valor'] - rep['cobertura_llave']['pct']) < 1e-6 and
      abs(k5.loc['pct_nombre_validado', 'valor'] - rep['pct_nombre_validado']['global']['pct']) < 1e-6)
print('huérfanos izquierda:', rep['huerfanos']['izquierda'], rep['huerfanos']['causas'], '| derecha:', rep['huerfanos']['derecha_barrios_sin_construcciones'])


                          numerador  denominador     valor
grupo                                                     
cobertura_llave              5140.0       5146.0  0.998834
cobertura_estrato_valido     5138.0       5146.0  0.998445
pct_nombre_validado          4807.0       5140.0  0.935214
umbral: 0.8 -> pasa | llave_confianza: alta
K5 de Gold coincide con quality_report.json: True
huérfanos izquierda: 6 {'COMUNA o BARRIO en blanco': 2, 'código ausente del CSV 2015': 4} | derecha: 295


## 3. Resultados de las pruebas

In [3]:
"""
Acá corremos pytest desde el notebook y mostramos el resumen. Son pruebas sin red: verifican el manifiesto y los sha256 de Bronze, las
llaves y los tipos de Silver, la cobertura y los rangos de los KPI, los vectores de Wilson y de Newcombe, y una regresión del defecto
que encontramos en el recálculo independiente de Gold, cuando Mantel-Haenszel contaba construcciones sin datos satelitales. También hay una prueba que obliga a que el
README tenga las mismas cifras que kpis_gold.csv. La salida debería ser todo en verde, sin pruebas saltadas. Que pasen no garantiza
que los datos sean verdaderos, solo que el pipeline hace lo que documentamos.
"""
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '--no-header', '--color=no'], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-900:]); print('código de salida:', r.returncode)
assert r.returncode == 0 and 'skipped' not in r.stdout and 'xfail' not in r.stdout


........................................                                 [100%]
40 passed in 1.67s

código de salida: 0


## 4. KPI y respuesta a la pregunta de negocio

In [4]:
"""
Aquí volvemos a mostrar los KPI principales y, justo debajo, la respuesta a la pregunta de negocio tal como quedó en el README, con la
estructura de seis frases que acordamos: no se detecta diferencia, el rango compatible, que eso no equivale a que el estrato no influya,
la inestabilidad del signo, la variación entre comunas y el peso de la Comuna 19. Leemos esas frases directamente del README para que
no haya dos versiones. La salida no concluye causalidad ni valida la hipótesis geotécnica del Avance 1; con este proxy y esta
muestra lo único que podemos decir es que no vemos diferencia, y eso es una afirmación sobre los datos, no sobre el mundo.
"""
print(kp[kp.kpi_id.isin(['K1', 'K2'])][['kpi_id', 'grupo', 'valor', 'numerador', 'denominador', 'ic95_inf', 'ic95_sup', 'pct_sin_datos', 'muestra_baja']].round(4).to_string(index=False))
readme = (ROOT / 'README.md').read_text(encoding='utf-8')
bloque = readme.split('### Respuesta a la pregunta de negocio')[1].split('### K1')[0]
print(bloque.strip())


kpi_id                    grupo  valor  numerador  denominador  ic95_inf  ic95_sup  pct_sin_datos  muestra_baja
    K1                estrato_1 0.0662       20.0        302.0    0.0433    0.1001         0.1065          True
    K1                estrato_2 0.0326        3.0         92.0    0.0112    0.0915         0.2698          True
    K1                estrato_3 0.0631       74.0       1173.0    0.0505    0.0785         0.0850         False
    K1                estrato_4 0.0692      145.0       2095.0    0.0591    0.0809         0.0759         False
    K1                estrato_5 0.0514       53.0       1031.0    0.0395    0.0666         0.0745         False
    K1                estrato_6 0.0000        0.0         11.0    0.0000    0.2588         0.0000          True
    K2             estratos_3-5 0.0633      272.0       4299.0    0.0564    0.0709         0.0781         False
    K2             estratos_1-2 0.0584       23.0        394.0    0.0392    0.0861         0.1509       

## 5. Composición del grupo de estratos 1-2

In [5]:
"""
Esta tabla la agregamos porque, al revisar K2, nos dimos cuenta de que el grupo de estratos 1 y 2 no está repartido por todas las comunas de la muestra:
casi todo cae en la Comuna 20, y un poco en la Comuna 18. Construimos la tabla de comuna por grupo con las construcciones con datos y
los severos, directamente desde gold_final.csv. La salida muestra que la Comuna 20 no tiene construcciones de los estratos 3 a 5 y que
solo la Comuna 18 tiene ambos grupos, por eso el ajuste por comuna no sirve. Esto no cambia ningún KPI, es una advertencia: K2 compara
en la práctica la Comuna 20 con el resto de la muestra, y no podemos separar el efecto del estrato del efecto del lugar.
"""
g = pd.read_csv(ROOT / 'data/gold/gold_final.csv', dtype={'comuna': str, 'codigo_barrio': str})
cc = g[~g.sin_datos & g.estrato.notna() & g.comuna.notna()].copy()
cc['grupo'] = cc.estrato.map(lambda s: '1-2' if s in (1, 2) else ('3-5' if s in (3, 4, 5) else '6'))
comp = cc.groupby(['comuna', 'grupo']).agg(n=('objectid', 'size'), k=('severo_estricto', 'sum')).unstack(fill_value=0)
print(comp)
t12 = cc[cc.grupo == '1-2'].groupby('comuna').agg(n=('objectid', 'size'), k=('severo_estricto', 'sum'))
print('\ngrupo 1-2 por comuna:', t12.to_dict('index'), '| % del grupo en C20:', round(t12.loc['20', 'n'] / t12.n.sum() * 100, 1))
print('comunas con ambos grupos:', [c for c in comp.index if comp.loc[c, ('n', '1-2')] > 0 and comp.loc[c, ('n', '3-5')] > 0])


          n             k        
grupo   1-2   3-5   6 1-2  3-5  6
comuna                           
03        0   550   0   0   39  0
09        0   184   0   0    7  0
10        0   541   0   0   35  0
17        0    87   0   0    4  0
18       52    44   0   3    5  0
19        0  2893  11   0  182  0
20      342     0   0  20    0  0

grupo 1-2 por comuna: {'18': {'n': 52, 'k': 3}, '20': {'n': 342, 'k': 20}} | % del grupo en C20: 86.8
comunas con ambos grupos: ['18']


## 6. Alcance frente al Proyecto 1 y limitaciones

In [6]:
"""
Para cerrar ubicamos nuestro trabajo frente a los cinco resultados clave del Proyecto 1: cubrimos KR4 con el proxy satelital, KR1 solo
de forma parcial, y dejamos KR2, KR3 y KR5 fuera, con la razón de cada uno. Debajo copiamos las limitaciones desde docs/gold/KPIs.md, que
es la fuente única. Dos avisos de alcance: el estrato es de 2015 y las imágenes de 2026, unos once años de distancia, así que los
resultados por estrato describen el estrato de 2015 asignado al barrio; y la población son 5.146 construcciones de 7 comunas dentro de
los polígonos de afectación preliminar, por lo que no extrapolamos a las construcciones de Cali. La salida no es un resultado nuevo sino
una descripción del alcance; el trabajo futuro es la amenaza sísmica oficial, la Comuna 2 y comparar con las inspecciones ATC-20.
"""
kr = pd.DataFrame([
    ('KR1', 'parcial', 'K3 y K6; solo 7 comunas'), ('KR2', 'no cubierto', 'no se usó amenaza sísmica oficial; trabajo futuro'),
    ('KR3', 'fuera de alcance', 'cualitativo, no es ETL'), ('KR4', 'cubierto con el proxy satelital', 'K1 y K2'),
    ('KR5', 'fuera de alcance', 'seguimiento, no es ETL')], columns=['KR', 'estado', 'evidencia'])
print(kr.to_string(index=False))
kpis_md = (ROOT / 'docs/gold/KPIs.md').read_text(encoding='utf-8')
sec = kpis_md.split('## Qué NO se puede concluir')[1].split('\n## ')[0]
print('\n' + '\n'.join(l for l in sec.splitlines() if re.match(r'^\d+\. ', l)))


 KR                          estado                                         evidencia
KR1                         parcial                           K3 y K6; solo 7 comunas
KR2                     no cubierto no se usó amenaza sísmica oficial; trabajo futuro
KR3                fuera de alcance                            cualitativo, no es ETL
KR4 cubierto con el proxy satelital                                           K1 y K2
KR5                fuera de alcance                            seguimiento, no es ETL

1. El proxy mide **cambio visible desde satélite**, no daño interno ni el de construcciones de varios pisos sin cambio visible; no sustituye la inspección ATC-20.
2. Solo hay **7 comunas** (03, 09, 10, 17, 18, 19, 20); la **Comuna 2 no está**. Las tasas describen esta muestra; no se extrapolan a las construcciones de Cali ni a la ciudad.
3. **E2 tiene ≈ 27 % sin datos** y la falta de datos no es aleatoria; sus cotas (2,4 %-29,4 % de severos) no permiten ordenar E2 frente a otros

## 7. Registro de ejecución

In [7]:
"""
Para terminar mostramos las últimas veinte líneas del registro del día, que es el archivo que escribe el pipeline en logs/. Lo hicimos
porque las pruebas dicen que el resultado es correcto, pero el registro cuenta cómo corrió: cuándo empezó y terminó cada etapa, cuántas
filas entraron y salieron y qué advertencias de calidad saltaron, por ejemplo las 6 construcciones sin llave o los grupos con muestra baja.
Ahí no aparecen errores. Esto no demuestra que no pueda fallar en otra máquina, solo que en esta corrida no falló, y el registro no
reemplaza las pruebas: sirve para auditar lo que pasó, no para verificar que esté bien.
"""
from datetime import datetime
log_hoy = ROOT / cfg['logging']['dir'] / datetime.now().strftime(cfg['logging']['file_pattern'])
lineas = log_hoy.read_text(encoding='utf-8').splitlines()
print(log_hoy.name, '|', len(lineas), 'líneas | líneas ERROR:', sum(' ERROR ' in l for l in lineas))
print('\n'.join(lineas[-20:]))


etl_20261111.log | 11 líneas | líneas ERROR: 0
2026-11-11 16:25:25 INFO etl.build_gold Inicio etapa gold
2026-11-11 16:25:26 INFO etl.build_gold Entrada: silver_integrated con 5146 construcciones
2026-11-11 16:25:26 INFO etl.build_gold Salida: gold_final=5146 filas | kpis_gold=23 filas (6 indicadores) | sensibilidades=107 filas
2026-11-11 16:25:26 WARNING etl.build_gold muestra baja (< 30 severos) en: estrato_1, estrato_2, estrato_6, estratos_1-2, comuna_09, comuna_17, comuna_18, comuna_20
2026-11-11 16:25:26 INFO etl.build_gold Fin etapa gold (1.10 s)
2026-11-11 16:25:26 INFO etl.load_database Inicio etapa load
2026-11-11 16:25:26 INFO etl.load_database silver_integrated: 5146 filas cargadas
2026-11-11 16:25:26 INFO etl.load_database gold_final: 5146 filas cargadas
2026-11-11 16:25:26 INFO etl.load_database kpis_gold: 23 filas cargadas
2026-11-11 16:25:26 INFO etl.load_database kpis_sensibilidad: 107 filas cargadas
2026-11-11 16:25:26 INFO etl.load_database Fin etapa load (0.47 s)
